In [10]:
import os
import time
import re
import json
import traceback
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ✅ Utilities
def safe_filename(variant):
    return re.sub(r'[<>:"/\\|?*]', "_", variant)

def double_encode(variant):
    return quote(quote(variant, safe=""), safe="")

# 🧪 Parser
def parse_cftr_functional_metrics(html_path, variant_name):
    with open(html_path, encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    result = {
        "variant_name": variant_name,
        "variant_type_text": None,
        "cftr_quantity_values": [],
        "cftr_function_values": [],
        "quantity_avg": None,
        "function_avg": None,
        "inferred_class": None,
        "final_call": None,
        "penetrance_text": None
    }

    h4_tag = soup.find("h4", string=re.compile(rf"Functional Testing for {variant_name}", re.I))
    if h4_tag:
        p_tag = h4_tag.find_next("p")
        if p_tag:
            result["variant_type_text"] = p_tag.get_text(" ", strip=True)

    table = soup.find("table", class_="missense_table")
    if table:
        for row in table.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) == 4 and variant_name in cells[0].text:
                qty = re.search(r"(\d+(?:\.\d+)?)", cells[2].text)
                func = re.search(r"(\d+(?:\.\d+)?)", cells[3].text)
                if qty:
                    result["cftr_quantity_values"].append(float(qty.group(1)))
                if func:
                    result["cftr_function_values"].append(float(func.group(1)))

    if result["cftr_quantity_values"]:
        result["quantity_avg"] = round(sum(result["cftr_quantity_values"]) / len(result["cftr_quantity_values"]), 2)
    if result["cftr_function_values"]:
        result["function_avg"] = round(sum(result["cftr_function_values"]) / len(result["cftr_function_values"]), 2)

    q = result["quantity_avg"]
    f = result["function_avg"]
    tag = result["variant_type_text"]

    if q == 0:
        result["inferred_class"] = "Class I"
    elif q is not None and f is not None:
        if q < 20 and f < 10:
            result["inferred_class"] = "Class II"
        elif q >= 80 and f < 10:
            result["inferred_class"] = "Class III"
        elif f < 20:
            result["inferred_class"] = "Class IV"
        elif f >= 80:
            result["inferred_class"] = "Likely non-CF"
        else:
            result["inferred_class"] = "Uncertain"
    elif tag and "nonsense variant" in tag.lower():
        result["inferred_class"] = "Class I"
    elif tag and "splice variant" in tag.lower():
        result["inferred_class"] = "Class I or V"

    history = soup.find("div", class_="drop_box")
    if history:
        for row in history.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) >= 2 and variant_name in cells[0].text:
                result["final_call"] = cells[1].text.strip()

    pop_tab = soup.find("div", id="population")
    if pop_tab:
        text = pop_tab.get_text(" ", strip=True)
        match = re.search(rf"{variant_name}.*?likely causes CF.*?\.", text, re.I)
        if match:
            result["penetrance_text"] = match.group(0)

    return result

# 🔁 Batch Scraping
df = pd.read_excel("CFTR2_25September2024.xlsx", sheet_name="CFTR2 variants by legacy name", skiprows=9)
variant_names = [v.strip() for v in df["Variant legacy name"].dropna().unique()]

SAVE_FOLDER = "cftr2_variant_pages"
os.makedirs(SAVE_FOLDER, exist_ok=True)
json_file = "cftr2_variant_classes.json"

if os.path.exists(json_file):
    with open(json_file, encoding="utf-8") as f:
        class_data_all = json.load(f)
else:
    class_data_all = {}

processed = set(class_data_all.keys())
failed = []
variant_log = []

# 🚀 Start Chrome
options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

# ☑️ Accept CFTR2 terms
print("🔐 Visiting CFTR2 welcome page...")
driver.get("https://cftr2.org/welcome")
time.sleep(1)

checkbox_ids = ["edit-education", "edit-individual", "edit-discuss", "edit-privacy"]
for box_id in checkbox_ids:
    try:
        checkbox = wait.until(EC.presence_of_element_located((By.ID, box_id)))
        driver.execute_script("arguments[0].scrollIntoView(true);", checkbox)
        if not checkbox.is_selected():
            checkbox.click()
        print(f"✅ Checked: {box_id}")
    except:
        print(f"⚠️ Could not find box: {box_id}")

submit_btn = wait.until(EC.element_to_be_clickable((By.ID, "edit-submit-terms")))
submit_btn.click()
print("✅ Submitted agreement.")

# 🔄 Process all variants
for variant in variant_names:
    variant_clean = variant.strip()

#### NOTE: This section is commented out to avoid reprocessing already handled variants.

# if variant_clean in processed:
#     print(f"⏭️ Skipping: {variant_clean}")
#     continue

    try:
        print(f"\n🔬 Processing: {variant_clean}")
        encoded = double_encode(variant_clean)
        url = f"https://cftr2.org/mutation/scientific/{encoded}"
        print(f"🔗 URL: {url}")
        driver.get(url)
        time.sleep(2)

        html_path = os.path.join(SAVE_FOLDER, f"{safe_filename(variant_clean)}.html")
        with open(html_path, "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        print(f"📄 Saved HTML to: {html_path}")

        info = parse_cftr_functional_metrics(html_path, variant_clean)
        class_data_all[variant_clean] = info
        variant_log.append([variant_clean, url, "Success"])
        print(f"🏷️ Inferred Class: {info['inferred_class']}")

        with open(json_file, "w", encoding="utf-8") as f:
            json.dump(class_data_all, f, indent=2)

    except Exception as e:
        print(f"❌ Error with {variant_clean}: {e}")
        failed.append(variant_clean)
        variant_log.append([variant_clean, url, f"Error: {str(e)}"])
        traceback.print_exc()

# 🛑 Shutdown
driver.quit()
with open("cftr2_variant_failed_log.txt", "w") as f:
    f.write("\n".join(failed))

with open("cftr2_variant_url_log.csv", "w", encoding="utf-8") as f:
    import csv
    writer = csv.writer(f)
    writer.writerow(["Variant", "URL", "Status"])
    writer.writerows(variant_log)

print("\n✅ Finished scraping all variants.")
print("🧪 Functional classification saved to cftr2_variant_classes.json")


🔐 Visiting CFTR2 welcome page...
✅ Checked: edit-education
✅ Checked: edit-individual
✅ Checked: edit-discuss
✅ Checked: edit-privacy
✅ Submitted agreement.

🔬 Processing: F508del
🔗 URL: https://cftr2.org/mutation/scientific/F508del
📄 Saved HTML to: cftr2_variant_pages\F508del.html
🏷️ Inferred Class: Class II

🔬 Processing: G542X
🔗 URL: https://cftr2.org/mutation/scientific/G542X
📄 Saved HTML to: cftr2_variant_pages\G542X.html
🏷️ Inferred Class: Class I

🔬 Processing: G551D
🔗 URL: https://cftr2.org/mutation/scientific/G551D
📄 Saved HTML to: cftr2_variant_pages\G551D.html
🏷️ Inferred Class: Class III

🔬 Processing: N1303K
🔗 URL: https://cftr2.org/mutation/scientific/N1303K


KeyboardInterrupt: 

In [3]:
import os
import time
import re
import json
import pandas as pd
import traceback
from bs4 import BeautifulSoup
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def safe_filename(name):
    return re.sub(r'[<>:"/\\|?*]', "_", name)

def double_encode(text):
    return quote(quote(text, safe=""), safe="")

def parse_cftr_functional_metrics(html_path, variant_name):
    with open(html_path, encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    result = {
        "variant_name": variant_name,
        "variant_type_text": None,
        "cftr_quantity_values": [],
        "cftr_function_values": [],
        "quantity_avg": None,
        "function_avg": None,
        "inferred_class": None,
        "final_call": None,
        "penetrance_text": None
    }

    # 🧬 Descriptor from <h4> + <p>
    for h in soup.find_all("h4"):
        header_text = h.get_text(" ", strip=True).lower()
        if "functional testing for" in header_text and variant_name.lower() in header_text:
            p_tag = h.find_next_sibling()
            while p_tag and p_tag.name != "p":
                p_tag = p_tag.find_next_sibling()
            if p_tag:
                result["variant_type_text"] = p_tag.get_text(" ", strip=True)
            break

    # 🧪 Functional metrics table
    table = soup.find("table", class_=re.compile("missense_table"))
    if table:
        for row in table.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) == 4 and any(variant_name in c.get_text() for c in cells):
                qty_match = re.search(r"(\d+(?:\.\d+)?)", cells[2].text)
                func_match = re.search(r"(\d+(?:\.\d+)?)", cells[3].text)
                if qty_match:
                    result["cftr_quantity_values"].append(float(qty_match.group(1)))
                if func_match:
                    result["cftr_function_values"].append(float(func_match.group(1)))

    if result["cftr_quantity_values"]:
        result["quantity_avg"] = round(sum(result["cftr_quantity_values"]) / len(result["cftr_quantity_values"]), 2)
    if result["cftr_function_values"]:
        result["function_avg"] = round(sum(result["cftr_function_values"]) / len(result["cftr_function_values"]), 2)

    # 🧠 Class inference by metrics and descriptor
    q = result["quantity_avg"]
    f = result["function_avg"]
    tag = (result["variant_type_text"] or "").lower()

    if q == 0:
        result["inferred_class"] = "Class I"
    elif q is not None and f is not None:
        if q < 20 and f < 10:
            result["inferred_class"] = "Class II"
        elif q >= 80 and f < 10:
            result["inferred_class"] = "Class III"
        elif f < 20:
            result["inferred_class"] = "Class IV"
        elif f >= 80:
            result["inferred_class"] = "Likely non-CF"
        else:
            result["inferred_class"] = "Uncertain"

    if result["inferred_class"] is None:
        if "nonsense" in tag or "frameshift" in tag or "insertion" in tag:
            result["inferred_class"] = "Class I"
        elif "no cftr protein" in tag or "not functionally tested" in tag:
            result["inferred_class"] = "Class I"
        elif "splice variant" in tag:
            result["inferred_class"] = "Class I or V"
        elif "in frame insertion" in tag or "deletion" in tag:
            result["inferred_class"] = "Class II"

    # 📜 Final call
    history = soup.find("div", class_="drop_box")
    if history:
        for row in history.find_all("tr"):
            cells = row.find_all("td")
            if len(cells) >= 2 and variant_name in cells[0].text:
                result["final_call"] = cells[1].text.strip()

    # 👥 Penetrance
    pop_tab = soup.find("div", id="population")
    if pop_tab:
        match = re.search(rf"{variant_name}.*?likely causes CF.*?\.", pop_tab.get_text(" ", strip=True), re.I)
        if match:
            result["penetrance_text"] = match.group(0)

    return result

# 🎯 Setup
df = pd.read_excel("CFTR2_25September2024.xlsx", sheet_name="CFTR2 variants by legacy name", skiprows=9)
variant_names = [v.strip() for v in df["Variant legacy name"].dropna().unique()]

SAVE_FOLDER = "cftr2_variant_pages"
os.makedirs(SAVE_FOLDER, exist_ok=True)
results_json = "cftr2_variant_classes.json"
class_data_all = {}

options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

try:
    print("🔐 Opening CFTR2 welcome page...")
    driver.get("https://cftr2.org/welcome")
    time.sleep(1)
    for box_id in ["edit-education", "edit-individual", "edit-discuss", "edit-privacy"]:
        try:
            box = wait.until(EC.presence_of_element_located((By.ID, box_id)))
            if not box.is_selected():
                box.click()
            print(f"✅ Checked: {box_id}")
        except:
            print(f"⚠️ Skipped: {box_id}")
    wait.until(EC.element_to_be_clickable((By.ID, "edit-submit-terms"))).click()
    print("✅ Terms accepted.")

    for variant in variant_names:
        try:
            print(f"\n🌐 Processing: {variant}")
            encoded = double_encode(variant)
            url = f"https://cftr2.org/mutation/scientific/{encoded}"
            driver.get(url)
            time.sleep(2)

            html_path = os.path.join(SAVE_FOLDER, f"{safe_filename(variant)}.html")
            with open(html_path, "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            print(f"📄 Saved HTML")

            info = parse_cftr_functional_metrics(html_path, variant)
            class_data_all[variant] = info
            print(f"🏷️ Inferred Class: {info['inferred_class']}")
        except Exception as e:
            print(f"❌ Error: {variant} → {e}")
            traceback.print_exc()

finally:
    driver.quit()
    print("🛑 Browser closed.")

# 💾 Save all results
with open(results_json, "w", encoding="utf-8") as f:
    json.dump(class_data_all, f, indent=2)

print(f"\n✅ All variant classifications saved to: {results_json}")


🔐 Opening CFTR2 welcome page...
✅ Checked: edit-education
✅ Checked: edit-individual
✅ Checked: edit-discuss
✅ Checked: edit-privacy
✅ Terms accepted.

🌐 Processing: F508del
📄 Saved HTML
🏷️ Inferred Class: Class II

🌐 Processing: G542X
📄 Saved HTML
🏷️ Inferred Class: Class I

🌐 Processing: G551D
📄 Saved HTML
🏷️ Inferred Class: Class III

🌐 Processing: N1303K
📄 Saved HTML
🏷️ Inferred Class: Class II

🌐 Processing: W1282X
📄 Saved HTML
🏷️ Inferred Class: Class I

🌐 Processing: R117H
📄 Saved HTML
🏷️ Inferred Class: Uncertain

🌐 Processing: 3849+10kbC->T
📄 Saved HTML
🏷️ Inferred Class: None

🌐 Processing: 621+1G->T
📄 Saved HTML
🏷️ Inferred Class: None

🌐 Processing: R553X
📄 Saved HTML
🏷️ Inferred Class: Class I

🌐 Processing: 2789+5G->A
📄 Saved HTML
🏷️ Inferred Class: Class IV

🌐 Processing: 1717-1G->A
📄 Saved HTML
🏷️ Inferred Class: None

🌐 Processing: CFTRdele2,3
📄 Saved HTML
🏷️ Inferred Class: Class I

🌐 Processing: D1152H
📄 Saved HTML
🏷️ Inferred Class: Uncertain

🌐 Processing: G85E
📄 S